# Задача 2. Фаза 2: Реализация VITS + Emotion Conditioning

**Участник 2** | Голосовой ассистент с эмоциональным синтезом речи

---

## Содержание
1. [2.1 Установка зависимостей](#2.1)
2. [2.2 Конфигурация](#2.2)
3. [2.3 DataLoader для DUSHA](#2.3)
4. [2.4 Модули VITS](#2.4)
5. [2.5 Loss-функции](#2.5)
6. [2.6 Training loop](#2.6)
7. [2.7 TensorBoard логирование](#2.7)
8. [2.8 Инференс](#2.8)
9. [2.9 Экспорт модели](#2.9)

<a id='2.1'></a>
## 2.1 Установка зависимостей

In [1]:
# --- Установка зависимостей ---
# Kaggle/Colab (Linux): apt-get ставит системный espeak-ng.
# На Windows эта строка упадёт, но благодаря "|| true" ячейка продолжит работу.
!apt-get install -y -qq espeak-ng > /dev/null 2>&1 || true
!pip install -q torch torchaudio tensorboard matplotlib numpy scipy librosa phonemizer

# Windows-only: поставьте espeak-ng вручную
# https://github.com/espeak-ng/espeak-ng/releases (добавьте в PATH).

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 24.3 MB/s eta 0:00:00


In [2]:
import os
import json
import math
import glob
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
import torchaudio
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

2026-05-13 15:27:58.051251: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778686078.450156      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778686078.557806      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778686079.585920      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778686079.585974      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778686079.585977      57 computation_placer.cc:177] computation placer alr

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
Device: cuda


<a id='2.2'></a>
## 2.2 Конфигурация модели

In [ ]:


import os as _os
from pathlib import Path as _Path


def _resolve_data_dir() -> str:
    """Находим корень DUSHA (папку с подпапками crowd_train / crowd_test)."""
 
    env = _os.environ.get("DUSHA_DATA_DIR")
    if env and (_Path(env) / "crowd_train").exists():
        return env

  
    kaggle_input = _Path("/kaggle/input")
    if kaggle_input.exists():
       
        try:
            for ct in kaggle_input.rglob("crowd_train"):
                if ct.is_dir() and (ct.parent / "crowd_test").exists():
                    print(f"[+] DUSHA найдена: {ct.parent}")
                    return str(ct.parent)
          
            for ct in kaggle_input.rglob("crowd_train"):
                if ct.is_dir():
                    print(f"[+] DUSHA (только train): {ct.parent}")
                    return str(ct.parent)
        except Exception as exc:
            print(f"[!] Ошибка при поиске DUSHA в /kaggle/input: {exc}")
        print("[!] /kaggle/input существует, но crowd_train не найдена. "
              "Задайте путь через os.environ['DUSHA_DATA_DIR'].")


    local = _Path.cwd() / "archive"
    if (local / "crowd_train").exists():
        return str(local)

   
    return r"D:\Tusur\3 курсу\3 курс 2 модуль\гпо\task2_tts\archive"


def _resolve_output_dir() -> str:
    if _Path("/kaggle/working").exists():
        return "/kaggle/working/vits_checkpoints"
    return str(_Path.cwd() / "vits_checkpoints")


def _resolve_log_dir() -> str:
    if _Path("/kaggle/working").exists():
        return "/kaggle/working/tensorboard_logs"
    return str(_Path.cwd() / "tensorboard_logs")


@dataclass
class VITSConfig:
    """Конфигурация модели VITS + Emotion Conditioning."""

    # --- Пути ---
    data_dir: str = field(default_factory=_resolve_data_dir) 
    output_dir: str = field(default_factory=_resolve_output_dir)
    log_dir: str = field(default_factory=_resolve_log_dir)

    # --- Аудио ---
    sample_rate: int = 22050
    n_fft: int = 1024
    hop_length: int = 256
    win_length: int = 1024
    n_mels: int = 80
    segment_size: int = 8192  # длина сегмента аудио для обучения (в сэмплах)

    # --- Фонемы ---
    n_vocab: int = 64       # размер фонемного словаря (espeak-ng для русского)
    pad_id: int = 0

    # --- Эмоции ---
    n_emotions: int = 4      
    emotion_dim: int = 128   # размерность emotion embedding

    # --- Text Encoder ---
    hidden_channels: int = 192
    filter_channels: int = 768
    n_heads: int = 2
    n_layers_enc: int = 6
    kernel_size_enc: int = 3
    p_dropout: float = 0.1

    # --- Posterior Encoder ---
    n_layers_posterior: int = 8
    kernel_size_posterior: int = 5
    dilation_rate: int = 1

    # --- Flow ---
    n_flows: int = 4
    n_layers_flow: int = 4
    kernel_size_flow: int = 5

    # --- Generator (HiFi-GAN) ---
    upsample_rates: list = field(default_factory=lambda: [8, 8, 2, 2])
    upsample_initial_channel: int = 512
    upsample_kernel_sizes: list = field(default_factory=lambda: [16, 16, 4, 4])
    resblock_kernel_sizes: list = field(default_factory=lambda: [3, 7, 11])
    resblock_dilation_sizes: list = field(default_factory=lambda: [[1, 3, 5], [1, 3, 5], [1, 3, 5]])

    # --- Duration Predictor ---
    kernel_size_dp: int = 3
    n_layers_dp: int = 3
    filter_channels_dp: int = 256

    # --- Discriminator ---
    periods: list = field(default_factory=lambda: [2, 3, 5, 7, 11])

    # --- Обучение ---
    batch_size: int = 8
    learning_rate: float = 2e-4
    betas: tuple = (0.8, 0.99)
    lr_decay: float = 0.999875
    max_steps: int = 100000
    warmup_steps: int = 5000
    checkpoint_interval: int = 2000
    log_interval: int = 100
    eval_interval: int = 2000
    fp16: bool = True
    num_workers: int = 4
    seed: int = 42

    # --- Loss weights ---
    lambda_kl: float = 0.1
    lambda_mel: float = 45.0
    lambda_dur: float = 1.0
    lambda_fm: float = 2.0
    lambda_adv: float = 1.0

    # --- KL Annealing ---
    kl_anneal_steps: int = 10000


config = VITSConfig()

# Воспроизводимость
torch.manual_seed(config.seed)
np.random.seed(config.seed)
random.seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)

print("Конфигурация загружена.")
print(f"  Emotion classes: {config.n_emotions}, dim: {config.emotion_dim}")
print(f"  Hidden channels: {config.hidden_channels}")
print(f"  Batch size: {config.batch_size}")
print(f"  Max steps: {config.max_steps}")

<a id='2.3'></a>
## 2.3 DataLoader для DUSHA

Загружает данные, подготовленные Участником 1:
- Аудио (22050 Hz, нормализованные)
- Фонемные последовательности
- Эмоциональные метки (0–3)

In [4]:
class MelSpectrogramExtractor:
    """Извлечение мел-спектрограмм из аудио."""

    def __init__(self, cfg: VITSConfig):
        self.cfg = cfg
        nyquist = cfg.sample_rate / 2
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            win_length=cfg.win_length,
            n_mels=cfg.n_mels,
            f_min=10.0,           # ← было 0.0
            f_max=nyquist,        # ← было 8000.0
            power=1.0,
            norm="slaney",
            mel_scale="slaney",
        )

    def __call__(self, audio: torch.Tensor) -> torch.Tensor:
        if getattr(self, "_device", None) != audio.device:
            self.mel_transform = self.mel_transform.to(audio.device)
            self._device = audio.device
        mel = self.mel_transform(audio)
        mel = torch.log(torch.clamp(mel, min=1e-5))
        return mel
mel_extractor = MelSpectrogramExtractor(config)
print("MelSpectrogramExtractor создан.")

MelSpectrogramExtractor создан.


In [5]:
# === Phonemizer — текст в фонемы ===

# Словарь фонем 


# Базовый набор символов espeak-ng для русского
_PAD = '_'
_BOS = '^'
_EOS = '$'
_SPACE = ' '


_PHONEMES = [
    'a', 'b', 'bʲ', 'd', 'dʲ', 'dʒ', 'e', 'f', 'fʲ', 'g', 'gʲ',
    'i', 'j', 'k', 'kʲ', 'l', 'lʲ', 'm', 'mʲ', 'n', 'nʲ', 'o',
    'p', 'pʲ', 'r', 'rʲ', 's', 'sʲ', 'ʃ', 'ʃʲ', 't', 'tʲ', 'ts',
    'tɕ', 'u', 'v', 'vʲ', 'x', 'xʲ', 'z', 'zʲ', 'ʒ',
    'ə', 'ɪ', 'ɵ', 'ʊ', 'ɛ', 'æ', 'ɨ', 'ɐ', 'ɫ',
    'ˈ', 'ˌ', 'ː',  
]

# Полный словарь
VOCAB = [_PAD, _BOS, _EOS, _SPACE] + _PHONEMES
SYMBOL_TO_ID = {s: i for i, s in enumerate(VOCAB)}
ID_TO_SYMBOL = {i: s for s, i in SYMBOL_TO_ID.items()}


def text_to_phoneme_ids(text: str) -> List[int]:
    """Конвертация текста в последовательность ID фонем через espeak-ng."""
    try:
        from phonemizer import phonemize
        from phonemizer.backend import EspeakBackend
        phonemes = phonemize(
            text,
            language='ru',
            backend='espeak',
            strip=True,
            preserve_punctuation=False,
            with_stress=True,
        )
    except Exception:
        # Fallback: простая посимвольная токенизация
        phonemes = text.lower()

    ids = [SYMBOL_TO_ID.get(_BOS, 1)]
    for ch in phonemes:
        if ch in SYMBOL_TO_ID:
            ids.append(SYMBOL_TO_ID[ch])
        elif ch == ' ':
            ids.append(SYMBOL_TO_ID.get(_SPACE, 3))
    ids.append(SYMBOL_TO_ID.get(_EOS, 2))
    return ids



EMOTION_MAP = {'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3}
EMOTION_NAMES = {v: k for k, v in EMOTION_MAP.items()}

print(f"Размер словаря фонем: {len(VOCAB)}")
print(f"Эмоции: {EMOTION_MAP}")

Размер словаря фонем: 58
Эмоции: {'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3}


In [6]:
# === Dataset: DUSHA (SberDevices) ===

import json as _json
import hashlib
import os
import random
from pathlib import Path

import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

# Избегаем проблем с file descriptors при многих воркерах
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except Exception:
    pass


# DUSHA -> модель (positive -> happy, other/NaN -> отбрасываем)
_DUSHA_EMO_REMAP = {
    "neutral":  "neutral",
    "angry":    "angry",
    "sad":      "sad",
    "positive": "happy",
}


def _batch_phonemize(texts, batch_size=512):
    """
    Батч-фонемизация через один persistent-процесс espeak-ng.
    Возвращает список строк с фонемами (IPA). При недоступности phonemizer
    возвращает исходные тексты (используется символьная токенизация).
    """
    try:
        from phonemizer.backend import EspeakBackend
        backend = EspeakBackend(
            "ru",
            preserve_punctuation=False,
            with_stress=True,
            language_switch="remove-flags",
        )
    except Exception as exc:
        print(f"[!] phonemizer недоступен ({exc}) — используем символьную токенизацию.")
        return texts

    out = []
    n = len(texts)
    try:
        from tqdm.auto import tqdm
        it = tqdm(range(0, n, batch_size), desc="phonemize", unit="batch")
    except Exception:
        it = range(0, n, batch_size)
    for i in it:
        chunk = texts[i : i + batch_size]
        try:
            out.extend(backend.phonemize(chunk, strip=True))
        except Exception as exc:
            print(f"[!] phonemize batch {i}: {exc}")
            out.extend(chunk)
    return out


def _text_to_ids_symbolic(text: str) -> list:
    """Fallback: побуквенная токенизация (когда espeak не сработал)."""
    ids = [SYMBOL_TO_ID.get(_BOS, 1)]
    for ch in text:
        if ch in SYMBOL_TO_ID:
            ids.append(SYMBOL_TO_ID[ch])
        elif ch == " ":
            ids.append(SYMBOL_TO_ID.get(_SPACE, 3))
    ids.append(SYMBOL_TO_ID.get(_EOS, 2))
    return ids


class DUSHADataset(Dataset):
    """
    Датасет DUSHA (SberDevices) для обучения VITS + Emotion Conditioning.

    Ожидаемая структура cfg.data_dir:
        <data_dir>/
            crowd_train/
                wavs/*.wav
                raw_crowd_train.jsonl
            crowd_test/
                wavs/*.wav
                raw_crowd_test.jsonl
    """

    def __init__(
        self,
        cfg: "VITSConfig",
        split: str = "train",
        min_duration: float = 0.8,
        max_duration: float = 12.0,
        max_samples=None,
        phoneme_cache_dir=None,
    ):
        self.cfg = cfg
        self.split = split
        self.mel_extractor = MelSpectrogramExtractor(cfg)

        split_dir = "crowd_train" if split == "train" else "crowd_test"
        self.root = os.path.join(cfg.data_dir, split_dir)
        jsonl_path = os.path.join(self.root, f"raw_{split_dir}.jsonl")

        if not os.path.exists(jsonl_path):
            print(f"[!] [{split}] Не найден {jsonl_path} — синтетика")
            self.data = self._generate_synthetic(200 if split == "train" else 20)
            return

        self.data = []
        skipped_emo = skipped_dur = skipped_text = 0
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                safe = line.replace(": NaN", ": null")
                try:
                    item = _json.loads(safe)
                except Exception:
                    continue

                raw_emo = item.get("speaker_emo") or item.get("annotator_emo")
                if raw_emo not in _DUSHA_EMO_REMAP:
                    skipped_emo += 1
                    continue
                emotion = _DUSHA_EMO_REMAP[raw_emo]

                dur = item.get("duration") or 0.0
                if dur < min_duration or dur > max_duration:
                    skipped_dur += 1
                    continue

                text = (item.get("speaker_text") or "").strip()
                if not text:
                    skipped_text += 1
                    continue

                rel_audio = item.get("audio_path", "").replace("/", os.sep)
                audio_path = os.path.join(self.root, rel_audio)

                self.data.append({
                    "audio": audio_path,
                    "text": text,
                    "emotion": emotion,
                    "duration": dur,
                })

                if max_samples is not None and len(self.data) >= max_samples:
                    break

        print(
            f"[{split}] Загружено {len(self.data)} записей.\n"
            f"    пропущено по эмоциям: {skipped_emo}, "
            f"по длительности: {skipped_dur}, без текста: {skipped_text}"
        )

       
        self._prephonemize(phoneme_cache_dir)

  
    def _prephonemize(self, cache_dir=None):
        texts = [it["text"] for it in self.data]
        if not texts:
            return

        cache_dir = cache_dir or self._default_cache_dir()
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        # Хэш по всем текстам + словарю — для инвалидации кэша
        h = hashlib.md5()
        for t in texts:
            h.update(t.encode("utf-8"))
            h.update(b"\0")
        h.update(_json.dumps(SYMBOL_TO_ID, ensure_ascii=False, sort_keys=True).encode())
        cache_file = Path(cache_dir) / f"phonemes_{self.split}_{h.hexdigest()[:12]}.json"

        if cache_file.exists():
            print(f"[+] Кэш фонем найден: {cache_file}")
            with open(cache_file, "r", encoding="utf-8") as f:
                all_ids = _json.load(f)
        else:
            print(f"[~] Предпосчёт фонем ({len(texts)} шт.)...")
            phon_strs = _batch_phonemize(texts)
            all_ids = []
            for p in phon_strs:
                # Собираем ids по токенам словаря (многосимвольные сначала)
                ids = [SYMBOL_TO_ID.get(_BOS, 1)]
                i = 0
                tokens = sorted(
                    [t for t in VOCAB if t not in (_PAD, _BOS, _EOS)],
                    key=lambda x: -len(x),
                )
                while i < len(p):
                    matched = False
                    for t in tokens:
                        if p.startswith(t, i):
                            ids.append(SYMBOL_TO_ID[t])
                            i += len(t)
                            matched = True
                            break
                    if not matched:
                        # пробел/пунктуация
                        if p[i] == " ":
                            ids.append(SYMBOL_TO_ID.get(_SPACE, 3))
                        i += 1
                ids.append(SYMBOL_TO_ID.get(_EOS, 2))
                if len(ids) <= 2:
                    # если фонемы пустые — символьный фолбэк
                    ids = _text_to_ids_symbolic(texts[len(all_ids)])
                all_ids.append(ids)
            try:
                with open(cache_file, "w", encoding="utf-8") as f:
                    _json.dump(all_ids, f)
                print(f"[+] Кэш сохранён: {cache_file}")
            except Exception as exc:
                print(f"[!] Не удалось сохранить кэш: {exc}")

        # Запись в записи
        for item, ids in zip(self.data, all_ids):
            item["phoneme_ids"] = ids

    @staticmethod
    def _default_cache_dir():
        if Path("/kaggle/working").exists():
            return "/kaggle/working/phoneme_cache"
        return str(Path.cwd() / "phoneme_cache")

   
    def _generate_synthetic(self, n: int) -> list:
        data = []
        emotions = list(EMOTION_MAP.keys())
        for i in range(n):
            data.append({
                "audio": None,
                "phoneme_ids": [random.randint(1, len(VOCAB) - 1)
                                for _ in range(random.randint(10, 50))],
                "emotion": emotions[i % len(emotions)],
                "text": f"Синтетический пример {i}",
            })
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Фонемы уже предпосчитаны в __init__
        phoneme_ids = item.get("phoneme_ids")
        if not phoneme_ids:
            phoneme_ids = _text_to_ids_symbolic(item.get("text", ""))
        phoneme_ids = torch.LongTensor(phoneme_ids)

        emotion_id = EMOTION_MAP.get(item["emotion"], 0)
        emotion = torch.LongTensor([emotion_id])

        audio_path = item.get("audio")
        if audio_path and os.path.exists(audio_path):
            try:
                wav, sr = torchaudio.load(audio_path)
            except Exception as exc:
                print(f"[!] wav load: {exc}")
                wav = torch.randn(self.cfg.sample_rate) * 0.01
                sr = self.cfg.sample_rate
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)
            if sr != self.cfg.sample_rate:
                wav = torchaudio.functional.resample(wav, sr, self.cfg.sample_rate)
            wav = wav[0]
        else:
            length = random.randint(self.cfg.sample_rate, self.cfg.sample_rate * 5)
            wav = torch.randn(length) * 0.1

        mel = self.mel_extractor(wav)

        return {
            "phoneme_ids": phoneme_ids,
            "phoneme_lengths": torch.LongTensor([len(phoneme_ids)]),
            "mel": mel,
            "mel_lengths": torch.LongTensor([mel.shape[1]]),
            "wav": wav,
            "wav_lengths": torch.LongTensor([len(wav)]),
            "emotion": emotion,
        }


def collate_fn(batch):
    max_phoneme_len = max(b["phoneme_ids"].shape[0] for b in batch)
    max_mel_len = max(b["mel"].shape[1] for b in batch)
    max_wav_len = max(b["wav"].shape[0] for b in batch)

    phoneme_ids = torch.zeros(len(batch), max_phoneme_len, dtype=torch.long)
    phoneme_lengths = torch.zeros(len(batch), dtype=torch.long)
    mels = torch.zeros(len(batch), batch[0]["mel"].shape[0], max_mel_len)
    mel_lengths = torch.zeros(len(batch), dtype=torch.long)
    wavs = torch.zeros(len(batch), max_wav_len)
    wav_lengths = torch.zeros(len(batch), dtype=torch.long)
    emotions = torch.zeros(len(batch), dtype=torch.long)

    for i, b in enumerate(batch):
        plen = b["phoneme_ids"].shape[0]
        mlen = b["mel"].shape[1]
        wlen = b["wav"].shape[0]

        phoneme_ids[i, :plen] = b["phoneme_ids"]
        phoneme_lengths[i] = plen
        mels[i, :, :mlen] = b["mel"]
        mel_lengths[i] = mlen
        wavs[i, :wlen] = b["wav"]
        wav_lengths[i] = wlen
        emotions[i] = b["emotion"][0]

    return {
        "phoneme_ids": phoneme_ids,
        "phoneme_lengths": phoneme_lengths,
        "mels": mels,
        "mel_lengths": mel_lengths,
        "wavs": wavs.unsqueeze(1),
        "wav_lengths": wav_lengths,
        "emotions": emotions,
    }


print(f"data_dir:   {config.data_dir}")
print(f"output_dir: {config.output_dir}")
print(f"log_dir:    {config.log_dir}")

train_dataset = DUSHADataset(config, split="train")
val_dataset   = DUSHADataset(config, split="val", max_samples=256)


_NW = min(2, config.num_workers)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=_NW,
    pin_memory=True,
    drop_last=True,
    persistent_workers=_NW > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

batch = next(iter(train_loader))
print("\nФормы тензоров в батче:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}")

data_dir:   /kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd
output_dir: /kaggle/working/vits_checkpoints
log_dir:    /kaggle/working/tensorboard_logs
[train] Загружено 827621 записей.
    пропущено по эмоциям: 263, по длительности: 2062, без текста: 77007
[~] Предпосчёт фонем (827621 шт.)...


phonemize:   0%|          | 0/1617 [00:00<?, ?batch/s]

[+] Кэш сохранён: /kaggle/working/phoneme_cache/phonemes_train_1d9d586ff80f.json
[val] Загружено 256 записей.
    пропущено по эмоциям: 0, по длительности: 0, без текста: 0
[~] Предпосчёт фонем (256 шт.)...


phonemize:   0%|          | 0/1 [00:00<?, ?batch/s]

[+] Кэш сохранён: /kaggle/working/phoneme_cache/phonemes_val_2053faa0da7d.json

Формы тензоров в батче:
  phoneme_ids: torch.Size([8, 62])
  phoneme_lengths: torch.Size([8])
  mels: torch.Size([8, 80, 646])
  mel_lengths: torch.Size([8])
  wavs: torch.Size([8, 1, 165375])
  wav_lengths: torch.Size([8])
  emotions: torch.Size([8])


<a id='2.4'></a>
## 2.4 Модули VITS

Полная реализация всех компонентов VITS с добавлением Emotion Embedding.

In [ ]:
# ==============================================================================
# 2.4.0 Вспомогательные модули
# ==============================================================================

class LayerNorm(nn.Module):
    """Layer norm по каналам (channel-first)."""
    def __init__(self, channels, eps=1e-5):
        super().__init__()
        self.channels = channels
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(channels))
        self.beta = nn.Parameter(torch.zeros(channels))

    def forward(self, x):
        # x: (B, C, T)
        x = x.transpose(1, -1)  # (B, T, C)
        x = F.layer_norm(x, (self.channels,), self.gamma, self.beta, self.eps)
        return x.transpose(1, -1)  # (B, C, T)


class WaveNetBlock(nn.Module):
    """Один блок WaveNet (dilated convolution + gated activation)."""

    def __init__(self, hidden_channels, kernel_size, dilation, gin_channels=0):
        super().__init__()
        padding = (kernel_size * dilation - dilation) // 2
        self.conv = nn.Conv1d(
            hidden_channels, 2 * hidden_channels,
            kernel_size, dilation=dilation, padding=padding
        )
        self.out_proj = nn.Conv1d(hidden_channels, hidden_channels, 1)

        if gin_channels > 0:
            self.cond_layer = nn.Conv1d(gin_channels, 2 * hidden_channels, 1)
        else:
            self.cond_layer = None

    def forward(self, x, x_mask=None, g=None):
        # x: (B, C, T)
        residual = x
        h = self.conv(x)

        if self.cond_layer is not None and g is not None:
            h = h + self.cond_layer(g)

    
        h_tanh, h_sigmoid = h.chunk(2, dim=1)
        h = torch.tanh(h_tanh) * torch.sigmoid(h_sigmoid)

        h = self.out_proj(h)
        if x_mask is not None:
            h = h * x_mask

        return residual + h


def sequence_mask(length, max_length=None):
    """Универсальная маска: поддерживает length любой формы.
    Для length.shape=(B,) -> (B, max_length).
    Для length.shape=(B, T_text) -> (B, T_text, max_length).
    """
    if max_length is None:
        max_length = length.max()
    max_length = int(max(max_length, 1))  # защита от 0/отрицательных
    x = torch.arange(max_length, dtype=length.dtype, device=length.device)
    return x < length.unsqueeze(-1)


def generate_path(duration, mask):
    """
    Строит one-hot путь выравнивания фонема↔mel из длительностей.
    duration: (B, 1, T_text) — сколько mel-фреймов на каждую фонему
    mask:     (B, 1, T_mel)  — маска mel-шкалы (1.0 там, где не padding)
    return:   (B, 1, T_text, T_mel) — матрица выравнивания
    """
    b, _, t_text = duration.shape
    t_mel = mask.shape[2]

    cum_duration = torch.cumsum(duration.squeeze(1), dim=-1)  # (B, T_text)
    path = sequence_mask(cum_duration, t_mel).to(duration.dtype)  # (B, T_text, T_mel)
    # Вычитаем сдвинутый cumsum -> окно конкретной фонемы
    path = path - F.pad(path, (0, 0, 1, 0))[:, :-1, :]
    # Добавляем ось heads и применяем mel-маску по T_mel
    path = path.unsqueeze(1) * mask.unsqueeze(2)  # (B,1,T_text,T_mel)*(B,1,1,T_mel)
    return path


print("Вспомогательные модули загружены.")

In [8]:
# ==============================================================================
# 2.4.1 Emotion Embedding — НОВЫЙ МОДУЛЬ
# ==============================================================================

class EmotionEmbedding(nn.Module):
    """
    Emotion Embedding Layer.
    Преобразует ID эмоции (0-3) в вектор фиксированной размерности.
    Этот вектор конкатенируется с выходом Text Encoder.
    """

    def __init__(self, n_emotions: int, emotion_dim: int, hidden_channels: int):
        super().__init__()
        self.embedding = nn.Embedding(n_emotions, emotion_dim)
        # Проекция: (hidden_channels + emotion_dim) -> hidden_channels
        self.proj = nn.Linear(hidden_channels + emotion_dim, hidden_channels)

    def forward(self, x: torch.Tensor, emotion_ids: torch.Tensor) -> torch.Tensor:
        """
        x: (B, hidden_channels, T) — выход Text Encoder
        emotion_ids: (B,) — ID эмоций
        return: (B, hidden_channels, T) — эмоционально-окрашенное представление
        """
        # Получаем emotion embedding: (B, emotion_dim)
        emo = self.embedding(emotion_ids)  # (B, emotion_dim)

        # Расширяем по временной оси: (B, emotion_dim, 1) -> broadcast
        emo = emo.unsqueeze(2).expand(-1, -1, x.shape[2])  # (B, emotion_dim, T)

        # Конкатенация: (B, hidden_channels + emotion_dim, T)
        combined = torch.cat([x, emo], dim=1)

        # Проекция обратно: (B, T, hidden+emo) -> (B, T, hidden) -> (B, hidden, T)
        combined = combined.transpose(1, 2)  # (B, T, C)
        out = self.proj(combined)             # (B, T, hidden)
        out = out.transpose(1, 2)             # (B, hidden, T)

        return out



_emo = EmotionEmbedding(4, 128, 192)
_x = torch.randn(2, 192, 10)
_ids = torch.LongTensor([0, 2])
_out = _emo(_x, _ids)
print(f"EmotionEmbedding: input {_x.shape} + emotion {_ids.shape} -> output {_out.shape}")
del _emo, _x, _ids, _out

EmotionEmbedding: input torch.Size([2, 192, 10]) + emotion torch.Size([2]) -> output torch.Size([2, 192, 10])


In [9]:
# ==============================================================================
# 2.4.2 Text Encoder (Transformer с relative positional encoding)
# ==============================================================================

class RelativeMultiHeadAttention(nn.Module):
    """Multi-head attention с relative positional encoding (как в Transformer-XL)."""

    def __init__(self, channels, n_heads, p_dropout=0.0, window_size=4):
        super().__init__()
        assert channels % n_heads == 0
        self.channels = channels
        self.n_heads = n_heads
        self.d_k = channels // n_heads
        self.window_size = window_size

        self.qkv = nn.Conv1d(channels, 3 * channels, 1)
        self.out_proj = nn.Conv1d(channels, channels, 1)
        self.dropout = nn.Dropout(p_dropout)

        # Relative position embedding
        self.rel_emb = nn.Parameter(torch.randn(n_heads, 2 * window_size + 1, self.d_k) * 0.01)

    def forward(self, x, mask=None):
        B, C, T = x.shape

        qkv = self.qkv(x).view(B, 3, self.n_heads, self.d_k, T)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]  # (B, heads, d_k, T)

        # Attention scores
        attn = torch.matmul(q.transpose(2, 3), k) / math.sqrt(self.d_k)  # (B, heads, T, T)

        # Relative positional bias
        rel_pos = self._get_relative_positions(T, x.device)
        rel_selected = self.rel_emb[:, rel_pos]  # (n_heads, T, T, d_k)
        rel_bias = torch.einsum("bhid,hijd->bhij", q.transpose(2, 3), rel_selected)
        attn = attn + rel_bias / math.sqrt(self.d_k)

        if mask is not None:
            attn = attn.masked_fill(mask.unsqueeze(1) == 0, torch.finfo(attn.dtype).min)

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v.transpose(2, 3))  # (B, heads, T, d_k)
        out = out.transpose(2, 3).contiguous().view(B, C, T)
        return self.out_proj(out)

    def _get_relative_positions(self, length, device):
        positions = torch.arange(length, device=device).unsqueeze(0) - torch.arange(length, device=device).unsqueeze(1)
        positions = positions.clamp(-self.window_size, self.window_size) + self.window_size
        return positions


class FFN(nn.Module):
    """Feed-Forward Network с Conv1d."""

    def __init__(self, in_channels, filter_channels, kernel_size, p_dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filter_channels, kernel_size, padding=kernel_size // 2)
        self.conv2 = nn.Conv1d(filter_channels, in_channels, kernel_size, padding=kernel_size // 2)
        self.dropout = nn.Dropout(p_dropout)

    def forward(self, x, x_mask):
        x = self.conv1(x * x_mask)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x * x_mask)
        return x * x_mask


class TransformerEncoderLayer(nn.Module):
    """Один слой Transformer Encoder."""

    def __init__(self, channels, filter_channels, n_heads, kernel_size, p_dropout):
        super().__init__()
        self.attn = RelativeMultiHeadAttention(channels, n_heads, p_dropout)
        self.norm1 = LayerNorm(channels)
        self.ffn = FFN(channels, filter_channels, kernel_size, p_dropout)
        self.norm2 = LayerNorm(channels)
        self.dropout = nn.Dropout(p_dropout)

    def forward(self, x, x_mask):
        # Self-attention
        residual = x
        x = self.norm1(x)
        x = self.attn(x, x_mask)
        x = self.dropout(x) + residual

        # FFN
        residual = x
        x = self.norm2(x)
        x = self.ffn(x, x_mask)
        x = self.dropout(x) + residual

        return x * x_mask


class TextEncoder(nn.Module):
    """
    Text Encoder: phoneme embeddings -> Transformer -> mean и log_var
    для prior distribution в VAE.
    """

    def __init__(self, cfg: VITSConfig):
        super().__init__()
        self.cfg = cfg

        # Phoneme embedding
        self.emb = nn.Embedding(cfg.n_vocab, cfg.hidden_channels, padding_idx=cfg.pad_id)
        nn.init.normal_(self.emb.weight, 0.0, cfg.hidden_channels ** -0.5)

        # Transformer layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(
                cfg.hidden_channels, cfg.filter_channels,
                cfg.n_heads, cfg.kernel_size_enc, cfg.p_dropout
            )
            for _ in range(cfg.n_layers_enc)
        ])

        # Проекция в prior: mean и log_var
        self.proj = nn.Conv1d(cfg.hidden_channels, 2 * cfg.hidden_channels, 1)

    def forward(self, x, x_lengths):
        """
        x: (B, T) — фонемные ID
        x_lengths: (B,)
        return: (x, m_p, logs_p, x_mask)
        """
        x = self.emb(x).transpose(1, 2)  # (B, C, T)

        x_mask = sequence_mask(x_lengths, x.shape[2]).unsqueeze(1).float()  # (B, 1, T)
        x = x * x_mask

        for layer in self.encoder_layers:
            x = layer(x, x_mask)

        # Prior statistics
        stats = self.proj(x) * x_mask
        m_p, logs_p = stats.split(self.cfg.hidden_channels, dim=1)

        return x, m_p, logs_p, x_mask


print("TextEncoder загружен.")

TextEncoder загружен.


In [ ]:
# ==============================================================================
# 2.4.3 Posterior Encoder (WaveNet-блоки)
# ==============================================================================

class PosteriorEncoder(nn.Module):
    """
    Posterior Encoder: mel -> z (латентное представление).
    Используется только при обучении для вычисления KL divergence.
    """

    def __init__(self, in_channels, hidden_channels, out_channels,
                 kernel_size, n_layers, dilation_rate=1, gin_channels=0):
        super().__init__()
        self.out_channels = out_channels

        self.pre = nn.Conv1d(in_channels, hidden_channels, 1)
        self.blocks = nn.ModuleList([
            WaveNetBlock(hidden_channels, kernel_size, dilation_rate ** (i % 4), gin_channels)
            for i in range(n_layers)
        ])
        self.proj = nn.Conv1d(hidden_channels, 2 * out_channels, 1)

    def forward(self, x, x_lengths, g=None):
        """
        x: (B, mel_channels, T_mel)
        return: (z, m_q, logs_q, y_mask)
        """
        x_mask = sequence_mask(x_lengths, x.shape[2]).unsqueeze(1).float()
        x = self.pre(x) * x_mask

        for block in self.blocks:
            x = block(x, x_mask, g)

        stats = self.proj(x) * x_mask
        m_q, logs_q = stats.split(self.out_channels, dim=1)
        logs_q = logs_q.clamp(min=-10.0, max=10.0)  # защита от FP16 overflow

        # Reparameterization trick
        z = m_q + torch.randn_like(m_q) * torch.exp(logs_q)
        z = z * x_mask

        return z, m_q, logs_q, x_mask


print("PosteriorEncoder загружен.")

In [11]:
# ==============================================================================
# 2.4.4 Normalizing Flow (ResidualCouplingLayer)
# ==============================================================================

class ResidualCouplingLayer(nn.Module):
    """Один слой residual coupling для normalizing flow."""

    def __init__(self, channels, hidden_channels, kernel_size, n_layers, gin_channels=0):
        super().__init__()
        self.half_channels = channels // 2

        self.pre = nn.Conv1d(self.half_channels, hidden_channels, 1)
        self.blocks = nn.ModuleList([
            WaveNetBlock(hidden_channels, kernel_size, 1, gin_channels)
            for _ in range(n_layers)
        ])
        self.proj = nn.Conv1d(hidden_channels, self.half_channels, 1)
        self.proj.weight.data.zero_()
        self.proj.bias.data.zero_()

    def forward(self, x, x_mask, g=None, reverse=False):
        x0, x1 = x.split(self.half_channels, dim=1)

        h = self.pre(x0) * x_mask
        for block in self.blocks:
            h = block(h, x_mask, g)
        m = self.proj(h)

        if not reverse:
            x1 = (x1 + m) * x_mask
        else:
            x1 = (x1 - m) * x_mask

        return torch.cat([x0, x1], dim=1)


class FlipLayer(nn.Module):
    """Переворот каналов между coupling layers."""
    def forward(self, x, *args, reverse=False, **kwargs):
        return torch.flip(x, dims=[1])


class ResidualCouplingBlock(nn.Module):
    """Блок из нескольких Residual Coupling Layers."""

    def __init__(self, channels, hidden_channels, kernel_size, n_layers, n_flows, gin_channels=0):
        super().__init__()
        self.flows = nn.ModuleList()
        for _ in range(n_flows):
            self.flows.append(ResidualCouplingLayer(
                channels, hidden_channels, kernel_size, n_layers, gin_channels
            ))
            self.flows.append(FlipLayer())

    def forward(self, x, x_mask, g=None, reverse=False):
        if not reverse:
            for flow in self.flows:
                x = flow(x, x_mask, g, reverse=False)
        else:
            for flow in reversed(self.flows):
                x = flow(x, x_mask, g, reverse=True)
        return x


print("ResidualCouplingBlock загружен.")

ResidualCouplingBlock загружен.


In [12]:
# ==============================================================================
# 2.4.5 Generator / Decoder (HiFi-GAN v1)
# ==============================================================================

class ResBlock(nn.Module):
    """Residual block для HiFi-GAN."""

    def __init__(self, channels, kernel_size, dilations):
        super().__init__()
        self.convs1 = nn.ModuleList()
        self.convs2 = nn.ModuleList()

        for d in dilations:
            self.convs1.append(nn.utils.weight_norm(
                nn.Conv1d(channels, channels, kernel_size, dilation=d,
                          padding=(kernel_size * d - d) // 2)
            ))
            self.convs2.append(nn.utils.weight_norm(
                nn.Conv1d(channels, channels, kernel_size, dilation=1,
                          padding=(kernel_size - 1) // 2)
            ))

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            residual = x
            x = F.leaky_relu(x, 0.1)
            x = c1(x)
            x = F.leaky_relu(x, 0.1)
            x = c2(x)
            x = x + residual
        return x

    def remove_weight_norm(self):
        for c in self.convs1:
            nn.utils.remove_weight_norm(c)
        for c in self.convs2:
            nn.utils.remove_weight_norm(c)


class Generator(nn.Module):
    """
    HiFi-GAN v1 Generator.
    z (latent) -> upsampling -> waveform
    """

    def __init__(self, cfg: VITSConfig):
        super().__init__()
        self.num_upsamples = len(cfg.upsample_rates)

        self.conv_pre = nn.utils.weight_norm(
            nn.Conv1d(cfg.hidden_channels, cfg.upsample_initial_channel, 7, padding=3)
        )

        self.ups = nn.ModuleList()
        ch = cfg.upsample_initial_channel
        for i, (u, k) in enumerate(zip(cfg.upsample_rates, cfg.upsample_kernel_sizes)):
            self.ups.append(nn.utils.weight_norm(
                nn.ConvTranspose1d(ch, ch // 2, k, stride=u, padding=(k - u) // 2)
            ))
            ch = ch // 2

        # Multi-receptive field fusion (MRF)
        self.resblocks = nn.ModuleList()
        ch = cfg.upsample_initial_channel
        for i in range(self.num_upsamples):
            ch = ch // 2
            for k, d in zip(cfg.resblock_kernel_sizes, cfg.resblock_dilation_sizes):
                self.resblocks.append(ResBlock(ch, k, d))

        self.conv_post = nn.utils.weight_norm(nn.Conv1d(ch, 1, 7, padding=3))

    def forward(self, x):
        """x: (B, hidden_channels, T) -> (B, 1, T * prod(upsample_rates))"""
        x = self.conv_pre(x)

        for i, up in enumerate(self.ups):
            x = F.leaky_relu(x, 0.1)
            x = up(x)

            # MRF: сумма выходов resblocks с разными kernel sizes
            xs = 0
            for j in range(len(self.resblocks) // self.num_upsamples):
                idx = i * (len(self.resblocks) // self.num_upsamples) + j
                xs = xs + self.resblocks[idx](x)
            x = xs / (len(self.resblocks) // self.num_upsamples)

        x = F.leaky_relu(x, 0.1)
        x = self.conv_post(x)
        x = torch.tanh(x)
        return x

    def remove_weight_norm(self):
        nn.utils.remove_weight_norm(self.conv_pre)
        for up in self.ups:
            nn.utils.remove_weight_norm(up)
        for rb in self.resblocks:
            rb.remove_weight_norm()
        nn.utils.remove_weight_norm(self.conv_post)


print("Generator (HiFi-GAN) загружен.")

Generator (HiFi-GAN) загружен.


In [ ]:
# ==============================================================================
# 2.4.6 Duration Predictor + Stochastic Duration Predictor
# ==============================================================================

class DurationPredictor(nn.Module):
    """Предсказание длительности фонем (детерминированный)."""

    def __init__(self, in_channels, filter_channels, kernel_size, n_layers, p_dropout, gin_channels=0):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for i in range(n_layers):
            ch_in = in_channels if i == 0 else filter_channels
            self.convs.append(nn.Conv1d(ch_in, filter_channels, kernel_size, padding=kernel_size // 2))
            self.norms.append(LayerNorm(filter_channels))

        self.proj = nn.Conv1d(filter_channels, 1, 1)
        self.dropout = nn.Dropout(p_dropout)

        if gin_channels > 0:
            self.cond = nn.Conv1d(gin_channels, in_channels, 1)
        else:
            self.cond = None

    def forward(self, x, x_mask, g=None):
        if self.cond is not None and g is not None:
            x = x + self.cond(g)

        for conv, norm in zip(self.convs, self.norms):
            x = conv(x * x_mask)
            x = norm(x)
            x = F.relu(x)
            x = self.dropout(x)

        return self.proj(x * x_mask) * x_mask


class StochasticDurationPredictor(nn.Module):
    """
    Стохастический предсказатель длительности.
    При обучении: использует реальные длительности.
    При инференсе: сэмплирует из выученного распределения.
    """

    def __init__(self, in_channels, filter_channels, kernel_size, p_dropout, n_flows=4, gin_channels=0):
        super().__init__()
        self.in_channels = in_channels

        self.pre = nn.Conv1d(in_channels, filter_channels, 1)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(3):
            self.convs.append(nn.Conv1d(filter_channels, filter_channels, kernel_size,
                                        padding=kernel_size // 2))
            self.norms.append(LayerNorm(filter_channels))

        self.proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.log_flow = nn.Conv1d(filter_channels, 2, 1)  # mean и log_var

        self.post_pre = nn.Conv1d(1, filter_channels, 1)
        self.post_proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.post_convs = nn.ModuleList()
        self.post_norms = nn.ModuleList()
        for _ in range(3):
            self.post_convs.append(nn.Conv1d(filter_channels, filter_channels, kernel_size,
                                              padding=kernel_size // 2))
            self.post_norms.append(LayerNorm(filter_channels))
        self.post_flow = nn.Conv1d(filter_channels, 2, 1)

        self.dropout = nn.Dropout(p_dropout)

        if gin_channels > 0:
            self.cond = nn.Conv1d(gin_channels, in_channels, 1)
        else:
            self.cond = None

    def forward(self, x, x_mask, w=None, g=None, reverse=False, noise_scale=1.0):
        """
        x: (B, C, T_text)
        w: (B, 1, T_text) — реальные длительности (только при обучении)
        При reverse=True (инференс): сэмплирует длительности.
        """
        if self.cond is not None and g is not None:
            x = x + self.cond(g)

        x = x.detach()  # stop gradient от text encoder
        h = self.pre(x)
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h * x_mask)
            h = norm(h)
            h = F.relu(h)
            h = self.dropout(h)

        h = self.proj(h * x_mask) * x_mask
        stats = self.log_flow(h)
        m, logs = stats.split(1, dim=1)

        if not reverse:
            # Training: posterior из реальных длительностей
            assert w is not None
            log_w = torch.log(w.float().clamp(min=1e-5)) * x_mask

            h_w = self.post_pre(log_w)
            for conv, norm in zip(self.post_convs, self.post_norms):
                h_w = conv(h_w * x_mask)
                h_w = norm(h_w)
                h_w = F.relu(h_w)
                h_w = self.dropout(h_w)
            h_w = self.post_proj(h_w * x_mask) * x_mask
            stats_w = self.post_flow(h_w)
            m_q, logs_q = stats_w.split(1, dim=1)

            # KL divergence для duration
            kl_dur = torch.sum(
                logs - logs_q - 0.5
                + 0.5 * ((log_w - m_q).pow(2) * torch.exp(-2 * logs_q))
                + 0.5 * (torch.exp(2 * (logs_q - logs)))
            ) / torch.sum(x_mask)

            return kl_dur

        else:
            # Inference: сэмплирование длительностей
            z = torch.randn_like(m) * noise_scale
            logs_clamped = torch.clamp(logs, max=5.0)  # защита от exp overflow
            log_w = m + z * torch.exp(logs_clamped)
            log_w = torch.clamp(log_w, max=10.0)       # exp(10)≈22000 фреймов макс
            w = torch.exp(log_w) * x_mask
            w = torch.nan_to_num(w, nan=1.0, posinf=50.0, neginf=0.0)
            w = torch.ceil(w)  # округление вверх
            return w


print("Duration Predictors загружены.")

In [14]:
# ==============================================================================
# 2.4.7 Discriminator (Multi-Period + Multi-Scale)
# ==============================================================================

class PeriodDiscriminator(nn.Module):
    """Дискриминатор для одного периода (MPD sub-discriminator)."""

    def __init__(self, period):
        super().__init__()
        self.period = period

        self.convs = nn.ModuleList([
            nn.utils.weight_norm(nn.Conv2d(1, 32, (5, 1), (3, 1), (2, 0))),
            nn.utils.weight_norm(nn.Conv2d(32, 128, (5, 1), (3, 1), (2, 0))),
            nn.utils.weight_norm(nn.Conv2d(128, 512, (5, 1), (3, 1), (2, 0))),
            nn.utils.weight_norm(nn.Conv2d(512, 1024, (5, 1), (3, 1), (2, 0))),
            nn.utils.weight_norm(nn.Conv2d(1024, 1024, (5, 1), 1, (2, 0))),
        ])
        self.conv_post = nn.utils.weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, (1, 0)))

    def forward(self, x):
        """
        x: (B, 1, T)
        return: (score, feature_maps)
        """
        fmaps = []
        B, C, T = x.shape

        # Reshape для периодической свёртки
        if T % self.period != 0:
            n_pad = self.period - (T % self.period)
            x = F.pad(x, (0, n_pad), 'reflect')
            T = T + n_pad
        x = x.view(B, C, T // self.period, self.period)  # (B, 1, T/p, p)

        for conv in self.convs:
            x = F.leaky_relu(conv(x), 0.1)
            fmaps.append(x)

        x = self.conv_post(x)
        fmaps.append(x)
        return x.flatten(1, -1), fmaps


class MultiPeriodDiscriminator(nn.Module):
    """Multi-Period Discriminator (MPD)."""

    def __init__(self, periods=None):
        super().__init__()
        periods = periods or [2, 3, 5, 7, 11]
        self.discriminators = nn.ModuleList([PeriodDiscriminator(p) for p in periods])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs = [], []
        fmap_rs, fmap_gs = [], []

        for d in self.discriminators:
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            y_d_gs.append(y_d_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


class ScaleDiscriminator(nn.Module):
    """Один дискриминатор для MSD."""

    def __init__(self, use_spectral_norm=False):
        super().__init__()
        norm_f = nn.utils.spectral_norm if use_spectral_norm else nn.utils.weight_norm

        self.convs = nn.ModuleList([
            norm_f(nn.Conv1d(1, 128, 15, 1, padding=7)),
            norm_f(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            norm_f(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            norm_f(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmaps = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), 0.1)
            fmaps.append(x)
        x = self.conv_post(x)
        fmaps.append(x)
        return x.flatten(1, -1), fmaps


class MultiScaleDiscriminator(nn.Module):
    """Multi-Scale Discriminator (MSD)."""

    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            ScaleDiscriminator(use_spectral_norm=True),
            ScaleDiscriminator(),
            ScaleDiscriminator(),
        ])
        self.downsamplers = nn.ModuleList([
            nn.Identity(),
            nn.AvgPool1d(4, 2, padding=2),
            nn.AvgPool1d(4, 2, padding=2),
        ])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs = [], []
        fmap_rs, fmap_gs = [], []

        for d, ds in zip(self.discriminators, self.downsamplers):
            y_ds = ds(y)
            y_hat_ds = ds(y_hat)
            y_d_r, fmap_r = d(y_ds)
            y_d_g, fmap_g = d(y_hat_ds)
            y_d_rs.append(y_d_r)
            y_d_gs.append(y_d_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


print("Discriminators загружены.")

Discriminators загружены.


In [ ]:
# ==============================================================================
# 2.4.8 VITS — основная модель (собирает все компоненты)
# ==============================================================================

class VITS(nn.Module):
    """
    VITS: Variational Inference with adversarial learning for end-to-end TTS.
    Модифицирована: добавлен EmotionEmbedding для управления эмоциями.
    """

    def __init__(self, cfg: VITSConfig):
        super().__init__()
        self.cfg = cfg

        # === Компоненты ===
        self.text_encoder = TextEncoder(cfg)

        # Emotion Embedding (НОВОЕ)
        self.emotion_emb = EmotionEmbedding(
            cfg.n_emotions, cfg.emotion_dim, cfg.hidden_channels
        )

        self.posterior_encoder = PosteriorEncoder(
            in_channels=cfg.n_mels,
            hidden_channels=cfg.hidden_channels,
            out_channels=cfg.hidden_channels,
            kernel_size=cfg.kernel_size_posterior,
            n_layers=cfg.n_layers_posterior,
        )

        self.flow = ResidualCouplingBlock(
            channels=cfg.hidden_channels,
            hidden_channels=cfg.hidden_channels,
            kernel_size=cfg.kernel_size_flow,
            n_layers=cfg.n_layers_flow,
            n_flows=cfg.n_flows,
        )

        self.generator = Generator(cfg)

        self.duration_predictor = DurationPredictor(
            in_channels=cfg.hidden_channels,
            filter_channels=cfg.filter_channels_dp,
            kernel_size=cfg.kernel_size_dp,
            n_layers=cfg.n_layers_dp,
            p_dropout=cfg.p_dropout,
        )

        self.stochastic_duration_predictor = StochasticDurationPredictor(
            in_channels=cfg.hidden_channels,
            filter_channels=cfg.filter_channels_dp,
            kernel_size=cfg.kernel_size_dp,
            p_dropout=cfg.p_dropout,
        )

    def forward(self, phoneme_ids, phoneme_lengths, mels, mel_lengths, emotions):
        """
        Прямой проход (обучение).
        """
        # 1. Text Encoder
        x, m_p, logs_p, x_mask = self.text_encoder(phoneme_ids, phoneme_lengths)

        # 2. Emotion Conditioning (НОВОЕ)
        x = self.emotion_emb(x, emotions)
        # m_p и logs_p тоже нужно обновить с учётом эмоции
        # Перепроецируем после emotion conditioning
        stats = self.text_encoder.proj(x) * x_mask
        m_p, logs_p = stats.split(self.cfg.hidden_channels, dim=1)
        logs_p = logs_p.clamp(min=-10.0, max=10.0)  # защита от FP16 overflow

        # 3. Posterior Encoder (из мел-спектрограммы)
        z, m_q, logs_q, y_mask = self.posterior_encoder(mels, mel_lengths)

        # 4. Flow: z_q -> z_p (для KL divergence)
        z_p = self.flow(z, y_mask)

        # 5. Monotonic Alignment Search (MAS)
        # Вычисляем выравнивание между текстом и мел-спектрограммой
        with torch.no_grad():
            # Negative log-likelihood под prior
            s_p = torch.exp(-2 * logs_p)  # (B, C, T_text)
            neg_cent = (
                -0.5 * math.log(2 * math.pi)
                - torch.sum(logs_p.unsqueeze(3), dim=1)  # (B, T_text, 1)
                - 0.5 * torch.sum(
                    s_p.unsqueeze(3) * (z_p.unsqueeze(2) - m_p.unsqueeze(3)) ** 2,
                    dim=1
                )  # (B, T_text, T_mel)
            )

            # Жадный MAS (упрощённый вариант)
            attn_mask = x_mask.unsqueeze(-1) * y_mask.unsqueeze(2)  # (B, 1, T_text, T_mel)
            attn = self._mas_greedy(neg_cent, attn_mask.squeeze(1))
            attn = attn.unsqueeze(1)  # (B, 1, T_text, T_mel)

        # Длительности из выравнивания
        w = attn.sum(3)  # (B, 1, T_text) — сколько mel-фреймов на фонему

        # 6. Duration loss
        l_dur = self.stochastic_duration_predictor(x, x_mask, w=w, reverse=False)

        # Expand prior по выравниванию
        m_p_expanded = torch.matmul(attn.squeeze(1).transpose(1, 2), m_p.transpose(1, 2)).transpose(1, 2)
        logs_p_expanded = torch.matmul(attn.squeeze(1).transpose(1, 2), logs_p.transpose(1, 2)).transpose(1, 2)

        # 7. Случайный сегмент для генератора
        z_slice, ids_slice = self._rand_slice_segments(z, mel_lengths)

        # 8. Generator: z -> audio
        o = self.generator(z_slice)

        return {
            'audio_hat': o,                 # сгенерированное аудио
            'ids_slice': ids_slice,          # индексы сегмента
            'z': z,                          # латентное (posterior)
            'z_p': z_p,                      # латентное через flow
            'm_p': m_p_expanded,             # prior mean (expanded)
            'logs_p': logs_p_expanded,       # prior log_var (expanded)
            'm_q': m_q,                      # posterior mean
            'logs_q': logs_q,                # posterior log_var
            'y_mask': y_mask,                # маска мел
            'l_dur': l_dur,                  # duration loss
        }

    def _mas_greedy(self, neg_cent, mask):
        """Векторизованный MAS на GPU (без Python-циклов по batch/T_text)."""
        B, T_text, T_mel = neg_cent.shape
        device = neg_cent.device
        dtype = neg_cent.dtype

        # DP: векторизован по batch и T_text, цикл только по T_mel
        log_p = torch.full((B, T_text, T_mel), -1e9, device=device, dtype=dtype)
        log_p[:, 0, 0] = neg_cent[:, 0, 0]

        for j in range(1, T_mel):
            prev_same = log_p[:, :, j - 1]                         # (B, T_text)
            prev_up = torch.full_like(prev_same, -1e9)
            prev_up[:, 1:] = log_p[:, :-1, j - 1]                  # shift
            log_p[:, :, j] = torch.maximum(prev_same, prev_up) + neg_cent[:, :, j]

        # Backtrack — векторизован по batch
        attn = torch.zeros(B, T_text, T_mel, device=device, dtype=dtype)
        idx = torch.full((B,), T_text - 1, dtype=torch.long, device=device)
        b_idx = torch.arange(B, device=device)
        for j in range(T_mel - 1, -1, -1):
            attn[b_idx, idx, j] = 1.0
            if j > 0:
                can_up = idx > 0
                prev_same = log_p[b_idx, idx, j - 1]
                up_idx = (idx - 1).clamp(min=0)
                prev_up = log_p[b_idx, up_idx, j - 1]
                go_up = can_up & (prev_up > prev_same)
                idx = idx - go_up.long()

        return attn * mask

    def _rand_slice_segments(self, z, lengths, segment_size=None):
        """Случайная нарезка сегментов из z для генератора (torch.randint)."""
        if segment_size is None:
            segment_size = self.cfg.segment_size // self.cfg.hop_length

        B, C, T = z.shape
        max_starts = (lengths.clamp(max=T).long() - segment_size).clamp(min=0)
        ids = (torch.rand(B, device=z.device) * (max_starts.float() + 1)).long().clamp(min=0)

        segments = torch.zeros(B, C, segment_size, device=z.device, dtype=z.dtype)
        for i in range(B):
            end = min(ids[i].item() + segment_size, T)
            seg_len = end - ids[i].item()
            if seg_len > 0:
                segments[i, :, :seg_len] = z[i, :, ids[i]:end]

        return segments, ids

    @torch.no_grad()
    def infer(self, phoneme_ids, phoneme_lengths, emotions, noise_scale=0.667, noise_scale_w=0.8):
        """
        Инференс: текст + эмоция -> аудио.
        """
        # 1. Text Encoder
        x, m_p, logs_p, x_mask = self.text_encoder(phoneme_ids, phoneme_lengths)

        # 2. Emotion Conditioning
        x = self.emotion_emb(x, emotions)
        stats = self.text_encoder.proj(x) * x_mask
        m_p, logs_p = stats.split(self.cfg.hidden_channels, dim=1)
        logs_p = logs_p.clamp(min=-10.0, max=10.0)  # защита от FP16 overflow

        # 3. Duration prediction (с защитой от NaN/inf)
        w = self.stochastic_duration_predictor(x, x_mask, reverse=True, noise_scale=noise_scale_w)
        w = torch.nan_to_num(w, nan=1.0, posinf=50.0, neginf=0.0)
        w = torch.clamp(w, min=0.0, max=100.0)
        w = torch.ceil(w).long()

        # Вычисляем длину выходной последовательности
        y_lengths = w.sum(dim=[1, 2]).long().clamp(min=1)
        y_max_length = max(y_lengths.max().item(), 1)
        y_mask = sequence_mask(y_lengths, y_max_length).unsqueeze(1).float()

        # 4. Expand prior по предсказанным длительностям
        attn_mask = x_mask.unsqueeze(-1) * y_mask.unsqueeze(2)
        path = generate_path(w.float(), y_mask)

        m_p_expanded = torch.matmul(path.squeeze(1).transpose(1, 2), m_p.transpose(1, 2)).transpose(1, 2)
        logs_p_expanded = torch.matmul(path.squeeze(1).transpose(1, 2), logs_p.transpose(1, 2)).transpose(1, 2)

        # 5. Sample z from prior
        z_p = m_p_expanded + torch.randn_like(m_p_expanded) * torch.exp(logs_p_expanded) * noise_scale

        # 6. Flow (reverse): z_p -> z
        z = self.flow(z_p, y_mask, reverse=True)

        # 7. Generator: z -> audio
        audio = self.generator(z * y_mask)

        return audio, y_lengths


# === Создание модели ===
model = VITS(config).to(device)
mpd = MultiPeriodDiscriminator(config.periods).to(device)
msd = MultiScaleDiscriminator().to(device)

# Подсчёт параметров
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"\n=== Параметры модели ===")
print(f"  VITS (генератор): {count_params(model):,}")
print(f"    - Text Encoder:     {count_params(model.text_encoder):,}")
print(f"    - Emotion Embedding:{count_params(model.emotion_emb):,}")
print(f"    - Posterior Enc:    {count_params(model.posterior_encoder):,}")
print(f"    - Flow:             {count_params(model.flow):,}")
print(f"    - Generator:        {count_params(model.generator):,}")
print(f"    - Duration Pred:    {count_params(model.duration_predictor):,}")
print(f"    - Stoch Dur Pred:   {count_params(model.stochastic_duration_predictor):,}")
print(f"  MPD: {count_params(mpd):,}")
print(f"  MSD: {count_params(msd):,}")
print(f"  ВСЕГО: {count_params(model) + count_params(mpd) + count_params(msd):,}")

<a id='2.5'></a>
## 2.5 Loss-функции

In [ ]:
# ==============================================================================
# Loss-функции для VITS
# ==============================================================================

def kl_loss(z_p, logs_q, m_p, logs_p, z_mask):
    """
    KL divergence между posterior и prior.
    z_p: (B, C, T) — latent из posterior, пропущенный через flow
    ВАЖНО: считаем в FP32 — exp() переполняется в FP16.
    """
    z_p = z_p.float()
    logs_q = logs_q.float()
    m_p = m_p.float()
    logs_p = logs_p.float()
    z_mask = z_mask.float()

    kl = logs_p - logs_q - 0.5 + 0.5 * (
        (z_p - m_p) ** 2 * torch.exp(-2.0 * logs_p)
        + torch.exp(2.0 * (logs_q - logs_p))
    )
    kl = torch.sum(kl * z_mask) / torch.sum(z_mask)
    return kl


def feature_matching_loss(fmap_r, fmap_g):
    """Feature matching loss между реальным и сгенерированным."""
    loss = 0
    for dr, dg in zip(fmap_r, fmap_g):
        for rl, gl in zip(dr, dg):
            loss += torch.mean(torch.abs(rl.float().detach() - gl.float()))
    return loss


def discriminator_loss(disc_real_outputs, disc_generated_outputs):
    """Discriminator loss (hinge loss)."""
    loss = 0
    for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
        r_loss = torch.mean((1 - dr.float()) ** 2)
        g_loss = torch.mean(dg.float() ** 2)
        loss += r_loss + g_loss
    return loss


def generator_adv_loss(disc_outputs):
    """Generator adversarial loss."""
    loss = 0
    for dg in disc_outputs:
        loss += torch.mean((1 - dg.float()) ** 2)
    return loss


def mel_reconstruction_loss(mel_extractor, y_hat, y, mel_lengths):
    """Mel reconstruction loss (FP32 для стабильности STFT)."""
    # Приводим к одной длине
    min_len = min(y_hat.shape[-1], y.shape[-1])
    y_hat = y_hat[:, :, :min_len].float()
    y = y[:, :, :min_len].float()

    mel_hat = mel_extractor(y_hat.squeeze(1))
    mel_real = mel_extractor(y.squeeze(1))

    min_mel_len = min(mel_hat.shape[-1], mel_real.shape[-1])
    loss = F.l1_loss(mel_hat[:, :, :min_mel_len], mel_real[:, :, :min_mel_len])
    return loss


print("Loss-функции загружены.")

<a id='2.6'></a>
## 2.6 Training Loop

Полный цикл обучения с:
- AdamW optimizer
- ExponentialLR scheduler
- Mixed precision (FP16)
- Чекпоинты каждые 5000 шагов
- Resume training между сессиями Kaggle

In [17]:
# ==============================================================================
# Optimizers & Schedulers
# ==============================================================================

optim_g = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    betas=config.betas,
    weight_decay=0.01
)
optim_d = optim.AdamW(
    list(mpd.parameters()) + list(msd.parameters()),
    lr=config.learning_rate,
    betas=config.betas,
    weight_decay=0.01
)

scheduler_g = optim.lr_scheduler.ExponentialLR(optim_g, gamma=config.lr_decay)
scheduler_d = optim.lr_scheduler.ExponentialLR(optim_d, gamma=config.lr_decay)

scaler = GradScaler(enabled=config.fp16)

print("Optimizers и schedulers созданы.")

Optimizers и schedulers созданы.


/tmp/ipykernel_57/3124072032.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=config.fp16)


In [18]:
# ==============================================================================
# Чекпоинты: сохранение и загрузка
# ==============================================================================

def save_checkpoint(step, model, mpd, msd, optim_g, optim_d, scheduler_g, scheduler_d, scaler, path):
    """Сохранение чекпоинта для resume training."""
    checkpoint = {
        'step': step,
        'model': model.state_dict(),
        'mpd': mpd.state_dict(),
        'msd': msd.state_dict(),
        'optim_g': optim_g.state_dict(),
        'optim_d': optim_d.state_dict(),
        'scheduler_g': scheduler_g.state_dict(),
        'scheduler_d': scheduler_d.state_dict(),
        'scaler': scaler.state_dict(),
        'config': config.__dict__,
    }
    torch.save(checkpoint, path)
    print(f"💾 Чекпоинт сохранён: {path} (step {step})")


def load_checkpoint(path, model, mpd, msd, optim_g, optim_d, scheduler_g, scheduler_d, scaler):
    """Загрузка чекпоинта для продолжения обучения."""
    if not os.path.exists(path):
        print(f"⚠️ Чекпоинт {path} не найден, обучение с нуля.")
        return 0

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    mpd.load_state_dict(checkpoint['mpd'])
    msd.load_state_dict(checkpoint['msd'])
    optim_g.load_state_dict(checkpoint['optim_g'])
    optim_d.load_state_dict(checkpoint['optim_d'])
    scheduler_g.load_state_dict(checkpoint['scheduler_g'])
    scheduler_d.load_state_dict(checkpoint['scheduler_d'])
    scaler.load_state_dict(checkpoint['scaler'])

    step = checkpoint['step']
    print(f"✅ Чекпоинт загружен: {path} (step {step})")
    return step


checkpoint_path = "/kaggle/input/models/killerkloyn/12/pytorch/default/1/checkpoint_1000.pt"
# Альтернатива: используйте latest.pt
# checkpoint_path = "/kaggle/input/models/killerkloyn/12/pytorch/default/1/latest.pt"

if os.path.exists(checkpoint_path):
    print(f"📂 Загружаем чекпоинт: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Загружаем состояния моделей
    model.load_state_dict(checkpoint['model'])
    mpd.load_state_dict(checkpoint['mpd'])
    msd.load_state_dict(checkpoint['msd'])
    
    # Загружаем состояния оптимизаторов (если есть)
    if 'optim_g' in checkpoint:
        optim_g.load_state_dict(checkpoint['optim_g'])
        optim_d.load_state_dict(checkpoint['optim_d'])
        scheduler_g.load_state_dict(checkpoint['scheduler_g'])
        scheduler_d.load_state_dict(checkpoint['scheduler_d'])
        scaler.load_state_dict(checkpoint['scaler'])
    
    global_step = checkpoint.get('step', 1000)  # если step нет, используем 1000
    print(f"✅ Чекпоинт загружен! Продолжаем с шага {global_step}")
else:
    print(f"⚠️ Чекпоинт {checkpoint_path} не найден, начинаем с нуля")
    global_step = 0

⚠️ Чекпоинт /kaggle/input/models/killerkloyn/12/pytorch/default/1/checkpoint_1000.pt не найден, начинаем с нуля


In [19]:
# ==============================================================================
# TensorBoard
# ==============================================================================

writer = SummaryWriter(config.log_dir)
print(f"TensorBoard: {config.log_dir}")
print(f"Запуск: tensorboard --logdir={config.log_dir}")

TensorBoard: /kaggle/working/tensorboard_logs
Запуск: tensorboard --logdir=/kaggle/working/tensorboard_logs


In [ ]:
# ==============================================================================
# ОСНОВНОЙ TRAINING LOOP
# ==============================================================================

def train_step(batch, step):
    """Один шаг обучения."""

    # Данные на GPU
    phoneme_ids = batch['phoneme_ids'].to(device)
    phoneme_lengths = batch['phoneme_lengths'].to(device)
    mels = batch['mels'].to(device)
    mel_lengths = batch['mel_lengths'].to(device)
    wavs = batch['wavs'].to(device)
    wav_lengths = batch['wav_lengths'].to(device)
    emotions = batch['emotions'].to(device)

    # =====================
    # Generator forward
    # =====================
    with autocast(enabled=config.fp16):
        outputs = model(phoneme_ids, phoneme_lengths, mels, mel_lengths, emotions)

        # Нарезаем соответствующий сегмент реального аудио
        seg_size = outputs['audio_hat'].shape[-1]
        y_segments = torch.zeros_like(outputs['audio_hat'])
        for i in range(wavs.shape[0]):
            start = outputs['ids_slice'][i] * config.hop_length
            end = start + seg_size
            if end <= wavs.shape[-1]:
                y_segments[i] = wavs[i, :, start:end]
            else:
                available = wavs.shape[-1] - start
                if available > 0:
                    y_segments[i, :, :available] = wavs[i, :, start:]

    # =====================
    # Discriminator step
    # =====================
    optim_d.zero_grad()

    with autocast(enabled=config.fp16):
        y_hat_detach = outputs['audio_hat'].detach()

        # MPD
        y_d_rs_mpd, y_d_gs_mpd, _, _ = mpd(y_segments, y_hat_detach)
        loss_disc_mpd = discriminator_loss(y_d_rs_mpd, y_d_gs_mpd)

        # MSD
        y_d_rs_msd, y_d_gs_msd, _, _ = msd(y_segments, y_hat_detach)
        loss_disc_msd = discriminator_loss(y_d_rs_msd, y_d_gs_msd)

        loss_disc = loss_disc_mpd + loss_disc_msd

    scaler.scale(loss_disc).backward()
    scaler.unscale_(optim_d)
    torch.nn.utils.clip_grad_norm_(list(mpd.parameters()) + list(msd.parameters()), 5.0)
    scaler.step(optim_d)

    # =====================
    # Generator step
    # =====================
    optim_g.zero_grad()

    with autocast(enabled=config.fp16):
        # Adversarial loss
        _, y_d_gs_mpd, fmap_rs_mpd, fmap_gs_mpd = mpd(y_segments, outputs['audio_hat'])
        _, y_d_gs_msd, fmap_rs_msd, fmap_gs_msd = msd(y_segments, outputs['audio_hat'])

        loss_adv = generator_adv_loss(y_d_gs_mpd + y_d_gs_msd)

        # Feature matching loss
        loss_fm = feature_matching_loss(fmap_rs_mpd + fmap_rs_msd, fmap_gs_mpd + fmap_gs_msd)

        # Mel reconstruction loss
        loss_mel = mel_reconstruction_loss(mel_extractor, outputs['audio_hat'], y_segments, mel_lengths)

        # KL divergence
        loss_kl = kl_loss(
            outputs['z_p'], outputs['logs_q'],
            outputs['m_p'], outputs['logs_p'],
            outputs['y_mask']
        )

        # Duration loss
        loss_dur = outputs['l_dur']

        # KL annealing: линейный рост 0→1 за kl_anneal_steps
        kl_weight = min(1.0, step / max(config.kl_anneal_steps, 1))

        # Total generator loss
        loss_gen = (
            config.lambda_adv * loss_adv
            + config.lambda_fm * loss_fm
            + config.lambda_mel * loss_mel
            + config.lambda_kl * kl_weight * loss_kl
            + config.lambda_dur * loss_dur
        )

    scaler.scale(loss_gen).backward()
    scaler.unscale_(optim_g)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    scaler.step(optim_g)

    scaler.update()

    return {
        'loss_gen': loss_gen.item(),
        'loss_disc': loss_disc.item(),
        'loss_adv': loss_adv.item(),
        'loss_fm': loss_fm.item(),
        'loss_mel': loss_mel.item(),
        'loss_kl': loss_kl.item(),
        'loss_dur': loss_dur.item(),
    }


print("Training step определён.")

In [ ]:
# ==============================================================================
# ЗАПУСК ОБУЧЕНИЯ
# ==============================================================================

print(f"\n{'='*60}")
print(f"НАЧАЛО ОБУЧЕНИЯ VITS + Emotion Conditioning")
print(f"{'='*60}")
print(f"Стартовый шаг: {global_step}")
print(f"Целевое количество шагов: {config.max_steps}")
print(f"Batch size: {config.batch_size}")
print(f"FP16: {config.fp16}")
print(f"{'='*60}\n")

model.train()
mpd.train()
msd.train()

epoch = 0
running_losses = {}

while global_step < config.max_steps:
    epoch += 1

    for batch in train_loader:
        if global_step >= config.max_steps:
            break

        # --- Train step ---
        losses = train_step(batch, global_step)
        global_step += 1

        # --- Accumulate losses (capped to prevent memory leak) ---
        for k, v in losses.items():
            if k not in running_losses:
                running_losses[k] = []
            running_losses[k].append(v)
            if len(running_losses[k]) > 300:
                running_losses[k] = running_losses[k][-200:]

        # --- Logging ---
        if global_step % config.log_interval == 0:
            avg_losses = {k: np.mean(v[-config.log_interval:]) for k, v in running_losses.items()}

            print(
                f"Step {global_step:>7d}/{config.max_steps} | "
                f"G: {avg_losses['loss_gen']:.3f} | "
                f"D: {avg_losses['loss_disc']:.3f} | "
                f"mel: {avg_losses['loss_mel']:.3f} | "
                f"kl: {avg_losses['loss_kl']:.3f} | "
                f"dur: {avg_losses['loss_dur']:.3f}"
            )

            # TensorBoard
            for k, v in avg_losses.items():
                writer.add_scalar(f'train/{k}', v, global_step)
            writer.add_scalar('train/lr', scheduler_g.get_last_lr()[0], global_step)

        # --- Eval (с защитой от крэша на ранних шагах) ---
        if global_step % config.eval_interval == 0:
            model.eval()
            try:
                with torch.no_grad():
                    val_batch = next(iter(val_loader))
                    phoneme_ids = val_batch['phoneme_ids'].to(device)
                    phoneme_lengths = val_batch['phoneme_lengths'].to(device)
                    emotions = val_batch['emotions'].to(device)

                    audio_hat, _ = model.infer(phoneme_ids[:1], phoneme_lengths[:1], emotions[:1])

                    writer.add_audio(
                        f'eval/audio_step{global_step}',
                        audio_hat[0].cpu(),
                        global_step,
                        sample_rate=config.sample_rate
                    )
            except Exception as e:
                print(f"[!] Eval error at step {global_step}: {e}")
            model.train()

        # --- Checkpoint (с авто-очисткой старых, хранит последние 2) ---
        if global_step % config.checkpoint_interval == 0:
            save_checkpoint(
                global_step, model, mpd, msd,
                optim_g, optim_d, scheduler_g, scheduler_d, scaler,
                os.path.join(config.output_dir, 'latest.pt')
            )
            save_checkpoint(
                global_step, model, mpd, msd,
                optim_g, optim_d, scheduler_g, scheduler_d, scaler,
                os.path.join(config.output_dir, f'checkpoint_{global_step}.pt')
            )
            # Удаляем старые чекпоинты, оставляем 2 последних
            import glob as _g
            ckpts = sorted(_g.glob(os.path.join(config.output_dir, 'checkpoint_*.pt')))
            for old_ckpt in ckpts[:-2]:
                try:
                    os.remove(old_ckpt)
                    print(f"🗑️ Удалён старый: {old_ckpt}")
                except Exception:
                    pass

        # --- LR decay (step-based, каждые 1000 шагов) ---
        if global_step % 1000 == 0:
            scheduler_g.step()
            scheduler_d.step()

    # Epoch-level scheduler убран — используем step-based (ниже в цикле)
    pass

print(f"\n{'='*60}")
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО! Финальный шаг: {global_step}")
print(f"{'='*60}")

writer.close()

<a id='2.8'></a>
## 2.8 Инференс

Функция для синтеза речи: текст + эмоция → аудио.

In [ ]:
# ==============================================================================
# Инференс: текст + эмоция -> аудио
# ==============================================================================

def synthesize(text: str, emotion: str = 'neutral', noise_scale: float = 0.667,
               noise_scale_w: float = 0.8) -> np.ndarray:
    """
    Синтез речи с заданной эмоцией.

    Args:
        text: Текст для синтеза (русский)
        emotion: 'neutral', 'happy', 'sad', 'angry'
        noise_scale: Шум для разнообразия (0 = детерминированный)
        noise_scale_w: Шум для длительностей

    Returns:
        numpy array с аудио-сигналом (22050 Hz)
    """
    model.eval()

    # Текст -> фонемы -> тензор
    phoneme_ids = text_to_phoneme_ids(text)
    phoneme_ids = torch.LongTensor([phoneme_ids]).to(device)
    phoneme_lengths = torch.LongTensor([phoneme_ids.shape[1]]).to(device)

    # Эмоция -> ID
    emotion_id = EMOTION_MAP.get(emotion, 0)
    emotions = torch.LongTensor([emotion_id]).to(device)

    # Инференс
    with torch.no_grad():
        audio, lengths = model.infer(
            phoneme_ids, phoneme_lengths, emotions,
            noise_scale=noise_scale, noise_scale_w=noise_scale_w
        )

    audio = audio[0, 0].cpu().numpy()
    return audio


# === Тест инференса (после обучения) ===
test_phrases = [
    "Привет, как у тебя дела?",
    "Сегодня прекрасный день!",
    "Мне грустно это слышать.",
    "Это совершенно неприемлемо!",
]

test_emotions = ['neutral', 'happy', 'sad', 'angry']

print("Тестовый инференс:")
fig, axes = plt.subplots(len(test_phrases), 1, figsize=(14, 3 * len(test_phrases)))

for i, (phrase, emotion) in enumerate(zip(test_phrases, test_emotions)):
    audio = synthesize(phrase, emotion)
    print(f"  [{emotion:>8}] \"{phrase}\" -> {len(audio)/config.sample_rate:.2f} сек")

    axes[i].plot(audio, color=['gray', 'green', 'blue', 'red'][i], alpha=0.7)
    axes[i].set_title(f'{emotion}: "{phrase}"')
    axes[i].set_ylabel('Amplitude')

axes[-1].set_xlabel('Sample')
plt.tight_layout()
plt.savefig('inference_test.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='2.9'></a>
## 2.9 Экспорт модели

In [ ]:
# ==============================================================================
# Экспорт модели для инференса
# ==============================================================================

def export_pytorch(model, path):
    """Экспорт в формат PyTorch (.pt) — только weights генератора."""
    model.eval()
    state = {
        'model': model.state_dict(),
        'config': config.__dict__,
        'vocab': SYMBOL_TO_ID,
        'emotion_map': EMOTION_MAP,
    }
    torch.save(state, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"✅ PyTorch модель сохранена: {path} ({size_mb:.1f} MB)")


def export_onnx(model, path):
    """Экспорт в ONNX для быстрого инференса на CPU."""
    model.eval()

    # Dummy inputs
    phoneme_ids = torch.LongTensor([[1, 5, 10, 15, 20, 2]]).to(device)
    phoneme_lengths = torch.LongTensor([6]).to(device)
    emotions = torch.LongTensor([0]).to(device)

    # Создаём wrapper для экспорта infer метода
    class VITSInferWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, phoneme_ids, phoneme_lengths, emotions):
            audio, lengths = self.model.infer(phoneme_ids, phoneme_lengths, emotions)
            return audio

    wrapper = VITSInferWrapper(model)

    try:
        torch.onnx.export(
            wrapper,
            (phoneme_ids, phoneme_lengths, emotions),
            path,
            input_names=['phoneme_ids', 'phoneme_lengths', 'emotions'],
            output_names=['audio'],
            dynamic_axes={
                'phoneme_ids': {0: 'batch', 1: 'seq_len'},
                'audio': {0: 'batch', 2: 'audio_len'},
            },
            opset_version=14,
        )
        size_mb = os.path.getsize(path) / 1e6
        print(f"✅ ONNX модель сохранена: {path} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"⚠️ ONNX экспорт не удался: {e}")
        print("   Это нормально — VITS содержит динамические операции.")
        print("   Используйте PyTorch формат (.pt) для инференса.")


# Экспорт
export_dir = os.path.join(config.output_dir, 'export')
os.makedirs(export_dir, exist_ok=True)

export_pytorch(model, os.path.join(export_dir, 'vits_emotion.pt'))
export_onnx(model, os.path.join(export_dir, 'vits_emotion.onnx'))

print(f"\n📦 Экспортированные модели: {export_dir}")

## Итоги Фазы 2

### Что реализовано:
- **VITS** с полной архитектурой (Text Encoder, Posterior Encoder, Flow, HiFi-GAN Generator)
- **Emotion Embedding** (4 класса → 128-dim вектор, конкатенация с Text Encoder)
- **Duration Predictor** (детерминированный + стохастический)
- **Discriminator** (Multi-Period + Multi-Scale)
- **Training loop** с FP16, чекпоинтами, TensorBoard, resume training
- **Инференс**: текст + эмоция → аудио
- **Экспорт**: .pt и ONNX

### Следующий шаг
→ **Notebook 3 (Фаза 3)**: Оценка модели (MCD, MOS, тест эмоций)